# STM32 Propeller Swing Control

## Model matching, LQR-inspired state feedback, and real-time embedded implementation

This notebook is the central technical reference for a one-degree-of-freedom propeller-driven swing controlled by an STM32F10x microcontroller. It connects the physical system, mathematical models, firmware, experimental evidence, telemetry path, and engineering limitations.

### Evidence labels used throughout

- **Verified from firmware** - directly observable in the C source or Simulink model.
- **Reported experimental result** - recorded during physical testing; raw logs are not available in this repository.
- **Computed from documented parameters** - reproducible arithmetic based on stated model coefficients.
- **Illustrative computation** - generated to explain behavior; not a physical measurement.
- **Uncertainty** - a fact that cannot be confirmed from the available artifacts.


## 1. System objective

The plant is a freely rotating horizontal arm with one propeller at each end. Differential thrust creates a net torque about the central pivot. The controller must move the arm to a commanded angle and settle it with limited overshoot and steady-state error.

The command is generated by a rotary potentiometer. An incremental encoder measures arm position. The firmware estimates velocity from successive encoder samples, advances a second-order reference model, applies a tuned four-state feedback law, and drives two motors in opposite directions.

### Functional requirements

1. Accept an operator-defined position command.
2. Measure arm position in real time.
3. Run the controller at a nominal 20 ms interval.
4. Generate complementary motor commands.
5. Support PWM and analog-output actuation paths.
6. Stream controller data to a host computer for visualization.


## 2. Physical system

![Numbered system overview](../assets/figures/01-system-overview.jpg)

**Numbered components**

1. Analog motor-drive electronics.
2. Propeller/thruster pair.
3. Start/mode pushbutton.
4. Rotary command potentiometer.
5. Bidirectional H-bridge path.
6. STM32 microcontroller and interface electronics.

The detailed hardware photograph below shows the STM32 evaluation board, labeled wiring, prototyping boards, terminal blocks, and the mechanical pivot assembly.

<p align="center"><img src="../assets/images/hardware-setup.jpeg" alt="STM32 control electronics" width="520"></p>


## 3. Hardware and I/O map

| Function | Peripheral | Pin(s) | Evidence |
|---|---|---:|---|
| Motor PWM | TIM3 CH1/CH2, full remap | PC6, PC7 | Verified from `Motor.c` |
| Analog motor commands | DAC CH1/CH2 | PA4, PA5 | Verified from `DAC.c` |
| Quadrature encoder | TIM4 | PB6, PB7 | Verified from `encoder.c` |
| Command potentiometer | ADC1 channel 1 | PA1 | Verified from `ADC.c` |
| Start/zero button | GPIO input | PA0 | Verified from `main.c` |
| Telemetry transmit | USART2 TX | PA2 | Verified from `uart.c` |

**Known conflict:** `uart.c` also configures PA1 as USART2 RX even though USART2 is enabled in transmit-only mode. Because `UART_Init()` is called after `ADCinit()`, that operation overwrites the analog GPIO mode configured for the potentiometer input.


## 4. Firmware execution model

The main function initializes peripherals, waits for the operator to press the start button, initializes the encoder at the chosen zero position, and enables the controller timer. The foreground loop is intentionally empty; control and telemetry run in timer interrupts.

```c
  GPIO_InitTypeDef GPIO_InitStructure;
  uint16_t timestep=20; //in millisec
  
  /* Configure blue button as input floating */
  RCC_APB2PeriphClockCmd(RCC_APB2Periph_GPIOA, ENABLE);
  GPIO_InitStructure.GPIO_Mode = GPIO_Mode_IN_FLOATING;
  GPIO_InitStructure.GPIO_Pin = GPIO_Pin_0;
  GPIO_Init(GPIOA, &GPIO_InitStructure);
  
  /* Initializing: */
  Motor_Init();
  ADCinit();
  DACinit();
  UART_Init();

  /* Initialize theta0: theta0 is updating when button is pushed*/
  while (GPIO_ReadInputDataBit(GPIOA,GPIO_Pin_0)==0);

  Encoder_Init(); //encoder is initialized after button is pushed
  clock_init();
  controller_interrupt_init(timestep);

  
  /* Main loop: */
  while(1); // Infinite loop
```

**Verified from firmware.** The nominal controller interval is configured as 20 ms. Exact timing cannot be confirmed without the missing clock-tree and target project configuration.


### Real-time control loop

![Closed-loop controller flow](../assets/figures/07-controller-flowchart.png)

The diagram follows the interrupt-level execution order: acquire the command and feedback, update the measured and reference states, evaluate the controller, map its output to PWM and analog commands, and drive the propeller pair.


## 5. Plant identification

A measured open-loop step response was approximated by the second-order transfer function

$$
G(s)=\frac{83.01}{s^2+13.88s+83.01}.
$$

For the standard denominator $s^2+2\zeta\omega_n s+\omega_n^2$,

$$
\omega_n=\sqrt{83.01}, \qquad
\zeta=\frac{13.88}{2\sqrt{83.01}}.
$$

A state-space realization consistent with the firmware matrices is

$$
\dot{x}_p=A_px_p+B_pu,\qquad y_p=C_px_p,
$$

$$
A_p=\begin{bmatrix}0&1\\-83.01&-13.88\end{bmatrix},\qquad
B_p=\begin{bmatrix}0\\1\end{bmatrix},\qquad
C_p=\begin{bmatrix}83.01&0\end{bmatrix}.
$$

The following values are **computed from the documented coefficients**, not re-estimated from raw data.


In [ ]:
import cmath
import math

a1 = 13.88
a0 = 83.01
omega_n = math.sqrt(a0)
zeta = a1 / (2.0 * omega_n)
disc = complex(a1 * a1 - 4.0 * a0, 0.0)
p1 = (-a1 + cmath.sqrt(disc)) / 2.0
p2 = (-a1 - cmath.sqrt(disc)) / 2.0

print(f"Natural frequency: {omega_n:.4f} rad/s")
print(f"Damping ratio:     {zeta:.4f}")
print(f"Plant poles:       {p1.real:.4f} {p1.imag:+.4f}j, {p2.real:.4f} {p2.imag:+.4f}j")


Natural frequency: 9.1110 rad/s
Damping ratio:     0.7617
Plant poles:       -6.9400 +5.9031j, -6.9400 -5.9031j


### Reported open-loop response

![Open-loop response](../assets/figures/02-open-loop-response.png)

**Reported experimental result.** The plot records the input step and the measured angular response used to obtain the second-order approximation. The raw samples and identification script are unavailable, so the fitted coefficients cannot be independently reproduced from this repository.


### Firmware representation of the plant model

```c
// --- Matching LQR Controller Variables ---
// Controlled system: G(s) = 83.01/(s^2 + 13.88s + 83.01)
// Reference model: F(s) = 400/(s^2 + 40s + 400) (critically damped second-order filter)
float Ar[2][2] = { {0.0f, 1.0f}, {-400.0f, -40.0f} };
float Br[2] = { 0.0f, 1.0f };
float Cr[2] = { 400.0f, 0.0f };

// Extended system (4 states: x1, x2, xr1, xr2)
float AA[4][4] = {
    {0.0f, 1.0f, 0.0f, 0.0f},
    {-83.01f, -13.88f, 0.0f, 0.0f},
    {0.0f, 0.0f, 0.0f, 1.0f},
    {0.0f, 0.0f, -400.0f, -40.0f}
};
float BB[4] = {0.0f, 1.0f, 0.0f, 0.0f};
float CC[4] = {83.01f, 0.0f, -400.0f, 0.0f};
```

The matrices `AA`, `BB`, and `CC` document an extended plant/reference formulation but are not used by the runtime algorithm. The live controller uses measured encoder states, simulated reference-model states, the tuned gain vector, and feedforward.


## 6. Reference model

The desired closed-loop behavior is represented by

$$
F(s)=\frac{400}{s^2+40s+400}
=\frac{20^2}{(s+20)^2}.
$$

It is a critically damped second-order system with

$$
\omega_{n,r}=20\ \text{rad/s}, \qquad \zeta_r=1.
$$

A controllable realization is

$$
\dot{x}_r =
\begin{bmatrix}0&1\\-400&-40\end{bmatrix}x_r
+\begin{bmatrix}0\\1\end{bmatrix}r,
\qquad
y_r=\begin{bmatrix}400&0\end{bmatrix}x_r.
$$

The firmware advances this model with forward Euler at $T_s=0.02$ s:

$$
x_r[k+1]=(I+T_sA_r)x_r[k]+T_sB_rr[k]
=\begin{bmatrix}1&0.02\\-8&0.2\end{bmatrix}x_r[k]
+\begin{bmatrix}0\\0.02\end{bmatrix}r[k].
$$


```c
// Update the reference-model state using a digital simulation
void update_reference_state(float ref_input) {
    float xr_dot[2];
    xr_dot[0] = Ar[0][0]*xr[0] + Ar[0][1]*xr[1] + Br[0]*ref_input;
    xr_dot[1] = Ar[1][0]*xr[0] + Ar[1][1]*xr[1] + Br[1]*ref_input;
    xr[0] += 0.02f * xr_dot[0];
    xr[1] += 0.02f * xr_dot[1];
}
```

**Verified from firmware.** `Cr` is declared but unused; the controller directly includes `xr[0]` and `xr[1]` in the extended feedback vector.


In [ ]:
dt = 0.02
x1 = 0.0
x2 = 0.0
samples = {}
for k in range(26):
    t = k * dt
    if k in {0, 1, 2, 5, 10, 20, 25}:
        samples[t] = 400.0 * x1
    x1_dot = x2
    x2_dot = -400.0 * x1 - 40.0 * x2 + 1.0
    x1 += dt * x1_dot
    x2 += dt * x2_dot

print("Illustrative unit-step response of the firmware's Euler reference model")
for t, y in samples.items():
    print(f"t={t:>4.2f} s  y_r={y:.6f}")
print("Forward-Euler pole for the repeated continuous pole -20 rad/s: 1 + dt*(-20) = 0.6")


Illustrative unit-step response of the firmware's Euler reference model
t=0.00 s  y_r=0.000000
t=0.02 s  y_r=0.000000
t=0.04 s  y_r=0.160000
t=0.10 s  y_r=0.663040
t=0.20 s  y_r=0.953643
t=0.40 s  y_r=0.999476
t=0.50 s  y_r=0.999950
Forward-Euler pole for the repeated continuous pole -20 rad/s: 1 + dt*(-20) = 0.6


### Behavior before the reference filter

![Pre-filter response](../assets/figures/03-pre-filter-response.jpg)

**Reported experimental result.** The intermediate controller improves the open-loop behavior but still shows tracking error and substantial transients under changing commands. The reference model was introduced to shape the demanded trajectory and reduce abrupt motor effort.


## 7. State measurement and estimation

The firmware does not run a model-based observer. It forms a two-state approximation directly from encoder data:

$$
x_1[k]=n_{enc}[k],
\qquad
x_2[k]=\frac{n_{enc}[k]-n_{enc}[k-1]}{0.02}.
$$

```c
// Update the controlled-plant state from encoder data
void update_plant_state() {
    // x1 = encoder position; x2 = finite-difference velocity
    static int last_enc = 0;
    x[0] = (float)enc; // position
    x[1] = (float)(enc - last_enc) / 0.02f; // velocity (20 ms)
    last_enc = enc;
}
```

**Engineering interpretation.** This backward finite difference is inexpensive and suitable for a small MCU, but it amplifies encoder quantization and measurement noise. No derivative filter, wraparound correction, or observer is present.


## 8. Command mapping and angular calibration

The ADC command is centered around midscale and mapped to approximately $\pm450$ encoder-command counts:

$$
\theta_d=\operatorname{int}\left(\frac{ADC-2048}{2048}\,450\right).
$$

```c
    /* Sensor and time read */
    t=get_clock();
    pot=ReadADC();
    enc=Encoder_Pos();
    
    // Desired state from the potentiometer
    theta_desired = (int)(((float)pot-2048)/2048*450);
    
    // Update the controlled-plant state
    update_plant_state();

    // Update the reference-model state
    update_reference_state((float)theta_desired);
```

The physical calibration record relates a measured $22.62^\circ$ displacement to 500 counts:

$$
k_\theta=\frac{22.62^\circ}{500}=0.04524^\circ/\text{count}.
$$

![Angle calibration](../assets/figures/05-angle-calibration.jpg)

**Reported calibration.** The firmware itself operates in counts and counts per second; it does not convert the feedback state to degrees.


## 9. LQR formulation and implemented controller

The continuous-time LQR objective is

$$
J=\int_0^\infty \left(x_e^TQx_e+u^TRu\right)dt,
$$

where the extended state combines measured plant states and reference-model states:

$$
z=\begin{bmatrix}x_1&x_2&x_{r1}&x_{r2}\end{bmatrix}^T.
$$

The declared extended model is

$$
\dot z=A_ez+B_eu,\qquad e=C_ez,
$$

$$
A_e=\begin{bmatrix}
0&1&0&0\\
-83.01&-13.88&0&0\\
0&0&0&1\\
0&0&-400&-40
\end{bmatrix},\quad
B_e=\begin{bmatrix}0\\1\\0\\0\end{bmatrix},\quad
C_e=\begin{bmatrix}83.01&0&-400&0\end{bmatrix}.
$$

For a theoretical continuous-time design, $P$ solves

$$
A_e^TP+PA_e-PB_eR^{-1}B_e^TP+Q=0,
\qquad K=R^{-1}B_e^TP.
$$

A preliminary calculation produced

$$
K_{initial}=[2.3754,\ 1.528,\ 0,\ 0].
$$

Physical tuning produced the implemented vector

$$
K=[2.0,\ 1.25,\ -3.75,\ -0.6].
$$

The live control law is

$$
u[k]=-K
\begin{bmatrix}x_1&x_2&x_{r1}&x_{r2}\end{bmatrix}^T
+K_{ff}\theta_d,
\qquad K_{ff}=4.
$$

```c
// Experimentally tuned LQR-derived gain
float K[4] = {2.0f, 1.25f, -3.75f, -0.6f}; // retained experimental values

// States
float x[2] = {0.0f, 0.0f};    // plant state: [x1, x2]
float xr[2] = {0.0f, 0.0f};   // reference model state: [xr1, xr2]

// LQR weights
float Q = 0.0001f;
float R = 0.0001f;

// Feedforward term
float feedforward = 0.0f;
float Kff = 4.0f; // tuned feedforward gain

    // Form the extended state vector
    float x_ext[4] = { x[0], x[1], xr[0], xr[1] };
    // Evaluate u = -K*[x; xr] + feedforward
    float feedforward = Kff * theta_desired;
    float u = -K[0]*x_ext[0] - K[1]*x_ext[1] - K[2]*x_ext[2] - K[3]*x_ext[3] + feedforward;
```

**Important distinction.** The gain is described as LQR-derived and experimentally tuned. The repository does not contain the gain-calculation script or the final $Q$ and $R$ matrices. The runtime does not evaluate the declared extended matrices; calling the implementation a rigorously reproducible LQR synthesis would therefore overstate the available evidence.


## 10. Differential actuation

### PWM path

The scalar controller output is converted into two complementary timer commands:

$$
d_1=\operatorname{sat}_{[0,300]}\left(3\left(50-\frac{u}{20}\right)\right),
$$

$$
d_2=\operatorname{sat}_{[0,300]}\left(3\left(50+\frac{u}{20}\right)\right).
$$

### Analog path

The analog path uses a divisor of 27, converts the intermediate command to volts, and clamps both channels to the 0-3 V range:

$$
v_1=\operatorname{sat}_{[0,3]}\left(\frac{3}{100}\left(50-\frac{u}{27}\right)\right),
\qquad
v_2=\operatorname{sat}_{[0,3]}\left(\frac{3}{100}\left(50+\frac{u}{27}\right)\right).
$$

```c
    // Convert the controller output to complementary PWM commands
    u1 = 3.0f * (50.0f - u/20.0f);
    u2 = 3.0f * (50.0f + u/20.0f);
    
    // Saturation limit
    if (u1<0) u1=0;
    if (u2<0) u2=0;
    if (u1>300) u1=300;
    if (u2>300) u2=300;
    
    PWM_Motor_Drive((int)u1,(int)u2);
    u1 = 3.0f * (50.0f - u/27.0f);
    u2 = 3.0f * (50.0f + u/27.0f);
    u1analog=(u1)/100;
    u2analog=(u2)/100;
    // Analog saturation limits
    if (u1analog < 0) u1analog = 0;
    if (u1analog > 3) u1analog = 3;
    if (u2analog < 0) u2analog = 0;
    if (u2analog > 3) u2analog = 3;
    DAC_Motor_Drive(u1analog, u2analog);
    UART_transmit((int)u1,(int)u2, (int)x[0], theta_desired);
```

Both paths are executed on every control interrupt. The physical test configuration determines which motor interface actually controls the plant.


In [ ]:
def clamp(value, low, high):
    return max(low, min(high, value))

def commands(u):
    pwm1 = clamp(3.0 * (50.0 - u / 20.0), 0.0, 300.0)
    pwm2 = clamp(3.0 * (50.0 + u / 20.0), 0.0, 300.0)
    analog_raw1 = 3.0 * (50.0 - u / 27.0)
    analog_raw2 = 3.0 * (50.0 + u / 27.0)
    voltage1 = clamp(analog_raw1 / 100.0, 0.0, 3.0)
    voltage2 = clamp(analog_raw2 / 100.0, 0.0, 3.0)
    return pwm1, pwm2, voltage1, voltage2

print("Illustrative actuator mapping (not experimental data)")
print("      u |  PWM1  PWM2 | DAC1[V] DAC2[V]")
for u in [-1000, -500, 0, 500, 1000]:
    p1, p2, v1, v2 = commands(u)
    print(f"{u:7.0f} | {p1:5.1f} {p2:5.1f} |  {v1:5.3f}   {v2:5.3f}")


Illustrative actuator mapping (not experimental data)
      u |  PWM1  PWM2 | DAC1[V] DAC2[V]
  -1000 | 300.0   0.0 |  2.611   0.389
   -500 | 225.0  75.0 |  2.056   0.944
      0 | 150.0 150.0 |  1.500   1.500
    500 |  75.0 225.0 |  0.944   2.056
   1000 |   0.0 300.0 |  0.389   2.611


### Known DAC full-scale defect

The driver calculates a left-aligned 12-bit DAC word as

$$
D(v)=\operatorname{uint16}\!\left(\frac{v}{3}\,2^{16}\right).
$$

At exactly $v=3$ V, the intermediate result is 65536 and the `uint16_t` cast wraps it to zero. A valid 12-bit left-aligned full-scale word is approximately 65520 (`0xFFF0`). This is a verified source-code issue; it is documented here without changing the experimental firmware.


## 11. Experimental closed-loop results

### Analog actuation

![Analog closed-loop response](../assets/figures/04-analog-closed-loop-response.jpg)

**Reported experimental result**

| Metric | Value |
|---|---:|
| Rise time | 2.61 s |
| Peak time | 1.58 s |
| Steady-state error | 24.618 counts |
| Angular error using unrounded calibration | 1.1137 deg |

The analog path produced the smaller reported steady-state error and a smoother command signal.

### PWM actuation

![PWM closed-loop response](../assets/figures/06-pwm-closed-loop-response.jpg)

**Reported experimental result**

| Metric | Value |
|---|---:|
| Rise time | 2.065 s |
| Peak time | 1.435 s |
| Steady-state error | 68.322 counts |
| Angular error using 0.045 deg/count | 3.0745 deg |

The PWM path produced the faster reported response but more switching-related noise and a larger steady-state error.

### Interpretation boundary

The source plots are raster captures and no raw telemetry is available. The values above must be treated as recorded test results, not as independently recomputed metrics. The stated peak times are shorter than the stated rise times; the measurement definitions are unavailable and should not be inferred.


## 12. Controller-flow correspondence

The translated control-loop diagram in Section 4 matches the sequence in `TIM7_IRQHandler`: command acquisition, plant-state update, reference-model update, extended feedback, actuator conversion, and motor operation.


## 13. Telemetry and Simulink visualization

The STM32 transmits a framed packet every nominal 20 ms:

```text
0x0A | u1 int16 LE | u2 int16 LE | encoder int16 LE | desired int16 LE | 0x00
```

The Simulink model [`models/read_data_SWING.slx`](../models/read_data_SWING.slx) was saved with MATLAB R2017a and requires Instrument Control Toolbox. Its Serial Receive block uses:

- COM1
- 9600 baud, 8 data bits, no parity, one stop bit
- Little-endian `int16`
- Four values per frame
- LF header and NUL terminator
- 0.02 s sample time

It demultiplexes and displays `u1`, `u2`, `theta`, and `theta_desired`, and sends the two motor values and two position values to a two-input scope.

**Telemetry caveat:** `u1` and `u2` are transmitted after the analog-path mapping, so they are not the PWM compare values already written to TIM3 and are not the final 0-3 V values either.


### UART implementation

```c
    /*transmit start byte:*/
    USART_SendData(USART2,(uint8_t) '\n'); //start byte
    while(USART_GetFlagStatus(USART2, USART_FLAG_TXE) == RESET);          
    /*transmit a :*/
    USART_SendData(USART2,(uint8_t) a_byte1);
    while(USART_GetFlagStatus(USART2, USART_FLAG_TXE) == RESET);          
    USART_SendData(USART2,(uint8_t) a_byte2);
    while(USART_GetFlagStatus(USART2, USART_FLAG_TXE) == RESET);          
    /*transmit b :*/
    USART_SendData(USART2,(uint8_t) b_byte1);
    while(USART_GetFlagStatus(USART2, USART_FLAG_TXE) == RESET);          
    USART_SendData(USART2,(uint8_t) b_byte2);
    while(USART_GetFlagStatus(USART2, USART_FLAG_TXE) == RESET);          
    /*transmit c :*/
    USART_SendData(USART2,(uint8_t) c_byte1);
    while(USART_GetFlagStatus(USART2, USART_FLAG_TXE) == RESET);          
    USART_SendData(USART2,(uint8_t) c_byte2);
    while(USART_GetFlagStatus(USART2, USART_FLAG_TXE) == RESET);          
    /*transmit c :*/
    USART_SendData(USART2,(uint8_t) d_byte1);
    while(USART_GetFlagStatus(USART2, USART_FLAG_TXE) == RESET);          
    USART_SendData(USART2,(uint8_t) d_byte2);
    while(USART_GetFlagStatus(USART2, USART_FLAG_TXE) == RESET);          
    /*transmit end byte:*/
    USART_SendData(USART2,(uint8_t) '\0'); //end byte
    while(USART_GetFlagStatus(USART2, USART_FLAG_TXE) == RESET);          
```

At 9600 baud, transmitting the complete packet with blocking `TXE` waits consumes roughly half of a 20 ms control interval. TIM2 and TIM7 are configured at the same interrupt priority, so telemetry can introduce substantial control-loop jitter.


## 14. Physical demonstration

<p align="center">
  <video width="720" controls preload="metadata">
    <source src="../assets/media/controller-demo.mp4" type="video/mp4">
    Your notebook viewer does not support embedded HTML5 video.
  </video>
</p>

**Verified from video.** The operator initializes the apparatus and changes the command using the rotary potentiometer. The propeller-driven arm moves through several commanded angles and settles after each change while live traces update on the host display. The video demonstrates closed-loop setpoint tracking. It does not provide enough visual evidence to identify whether the active motor interface is PWM or analog, and it is not used for quantitative metric extraction.


## 15. Build and run status

The repository preserves the application-level firmware but is **not a standalone buildable STM32 project**. The following required items are absent:

- Exact STM32 part number and board target.
- Startup assembly and vector-table integration.
- Linker script and memory layout.
- System clock configuration.
- CMSIS and STM32F10x Standard Peripheral Library sources/headers.
- Compiler flags and an IDE, Make, CMake, or PlatformIO project file.

Consequently, no honest one-command build instruction can be provided. To reconstruct a build, create a project for the confirmed STM32F10x target, add the correct vendor library and startup/linker files, preserve the interrupt handler names, add `src/` and `include/`, and verify APB timer clocks before operating the hardware.

### Physical start sequence encoded in the firmware

1. Place the swing at the intended zero position.
2. Power the controller and motor electronics using the documented laboratory wiring.
3. Press the PA0 start/zero button.
4. The encoder is initialized and the timer-driven control loop starts.
5. Rotate the potentiometer to change the desired position.

These are behavioral observations from the source, not a substitute for a verified electrical safety procedure.


## 16. Known engineering limitations

### High impact

- PA1 is configured both as ADC input and USART2 RX GPIO.
- UART transmission busy-waits inside an interrupt at the same priority as the controller.
- The DAC full-scale conversion wraps 3 V to zero.
- Exact sample time cannot be verified without the target clock configuration.
- The firmware and reported results cannot be rebuilt from the repository alone.

### Control and measurement

- Velocity is an unfiltered finite difference.
- Encoder counter wraparound is not corrected.
- The `K` vector was physically tuned; its final synthesis data are unavailable.
- `AA`, `BB`, `CC`, `Cr`, `Q`, and `R` are declared but unused at runtime.
- Feedforward gain provenance is undocumented.
- PWM and DAC paths run simultaneously rather than through an explicit software mode selection.

### Safety and robustness

- No watchdog, fault-state machine, motor timeout, or explicit emergency shutdown is implemented in the application code.
- Invalid PWM input causes a permanent blocking loop.
- Shared telemetry state is not declared `volatile`.


## 17. Conclusions

The project demonstrates a complete embedded control chain: physical plant identification, second-order reference shaping, an LQR-inspired tuned feedback law, real-time STM32 execution, differential propeller actuation, physical experiments, and serial visualization.

The strongest verified contribution is the integration of sensing, reference-model simulation, state feedback, two actuator interfaces, and telemetry on a resource-constrained microcontroller. The experimental record supports the qualitative conclusion that PWM responds faster while analog actuation tracks more smoothly and accurately in this setup.

The main remaining engineering work is reproducibility and robustness: restore the exact target project and vendor dependencies, resolve the pin and DAC defects, decouple telemetry from the control interrupt, document the timing configuration, and preserve raw experimental data for independent analysis.


## Appendix A. Source traceability

| Topic | Primary artifact |
|---|---|
| Startup and control timer | [`src/main.c`](../src/main.c) |
| Reference model and controller | [`src/stm32f10x_it.c`](../src/stm32f10x_it.c) |
| PWM generation | [`src/Motor.c`](../src/Motor.c) |
| Analog output | [`src/DAC.c`](../src/DAC.c) |
| Encoder measurement | [`src/encoder.c`](../src/encoder.c) |
| Potentiometer ADC | [`src/ADC.c`](../src/ADC.c) |
| UART setup and shared data | [`src/uart.c`](../src/uart.c) |
| Host visualization | [`models/read_data_SWING.slx`](../models/read_data_SWING.slx) |
| Physical demonstration | [`assets/media/controller-demo.mp4`](../assets/media/controller-demo.mp4) |

## Appendix B. Evidence and reproducibility policy

- No synthetic trajectory is presented as an experiment.
- Numerical model characteristics are derived only from documented coefficients.
- Experimental metrics retain their reported status because raw samples are unavailable.
- Hardware and firmware uncertainties are stated explicitly rather than resolved by assumption.
